In [42]:
#Imports
import pandas as pd
import numpy as np

In [43]:
#Reading all files
df_payments = pd.read_csv("data/payments.csv")
df_buyers = pd.read_csv("data/buyers.csv")
df_merchants = pd.read_csv("data/merchants.csv")

In [44]:
#Determine the fraud label based on chargeback_timestamp
df_payments["is_fraud"] = df_payments["chargeback_timestamp"].notna().astype(int)

In [45]:
#merge the datasets
merged = pd.merge(df_payments,df_buyers, on='id', how='left')
merged = pd.merge(merged,df_merchants, on='id', how='left')
merged.head(5)

,id,merchant_id,buyer_id,transaction_timestamp,payment_amount,chargeback_timestamp,is_fraud,country_x,country_y,category
0,547601f8-6288-403d-8711-0d4e752f231c,2512b0ea-644d-4440-801d-63ca077a3ea0,6bbc28bd-49b8-424d-89c5-4239d5147e4e,2022-05-22 09:55:54,6352,NaN,0,NaN,NaN,NaN
1,4e069be8-e01b-4aee-bf1b-f3ddfc8cde6d,696b3ae8-a375-459b-ac76-cb17e15aec39,d4618c41-c414-4b4f-b439-e4f70b391496,2022-08-08 20:41:07,2810,NaN,0,NaN,NaN,NaN
2,bd64067f-5db1-4614-88ee-e678bad2ccce,5032c286-4189-4df1-8d8f-850e3ff44411,b1de3a9b-3688-4fc7-8c1d-87f669d9243e,2022-07-10 18:20:25,5226,NaN,0,NaN,NaN,NaN
3,2bebdfe8-f421-4ac5-9706-76938f0c0f64,0ae96c8a-1a2a-4070-8439-1d8e1a741238,e45a8c36-5720-4125-88b8-10e4c745fad4,2022-03-02 18:05:47,725,2022-03-10 03:24:40.790145,1,NaN,NaN,NaN
4,393f61b7-e789-41f7-9727-2e7d629e87d0,737032c5-6935-47bb-af7a-2d1e455a77cf,0cba1762-b083-4747-bf28-416906b29d2b,2022-02-12 09:08:37,2437,NaN,0,NaN,NaN,NaN


In [46]:
# Convert timestamp to datetime and sort
merged['transaction_timestamp'] = pd.to_datetime(merged['transaction_timestamp'])

#sorting
merged_sorted = merged.sort_values('transaction_timestamp').reset_index(drop=True)

In [47]:
#Merchant fraud rate
merged_sorted['merchant_fraud_rate'] = (
    merged_sorted.groupby('merchant_id')['is_fraud']
    .expanding().mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

In [48]:
merged_sorted['merchant_fraud_rate'] = merged_sorted['merchant_fraud_rate'].fillna(0)

In [49]:
merged_sorted.head(5)

,id,merchant_id,buyer_id,transaction_timestamp,payment_amount,chargeback_timestamp,is_fraud,country_x,country_y,category,merchant_fraud_rate
0,0ef41176-3ec5-4ab6-9c37-bb793ec75e54,977ce629-3cbe-43ec-878f-cf1c2aebe85d,e5737c15-ffcd-4111-a0b0-d6746abf7ab7,2022-01-01 00:22:09,3469,2022-01-07 00:30:10.855945,1,NaN,NaN,NaN,0.059829
1,596a94fa-7398-4b30-a0d8-cc518a8485c0,951bce36-de7c-4081-a97d-2fd80dab9174,ea546275-cf52-4ea7-80c9-f243bfeea139,2022-01-01 00:32:11,321,NaN,0,NaN,NaN,NaN,0.111111
2,1227adb3-b15b-46f8-9d15-71d0071019b8,af9ccc42-0a10-4acb-9a78-2ca0fe889cd9,b3ed51d9-8dbd-4e46-bfd3-b31217e8a033,2022-01-01 00:55:13,622,2022-01-13 11:51:50.293075,1,NaN,NaN,NaN,0.031250
3,1d6c373e-167a-48bd-a748-a48ca76a5afe,951bce36-de7c-4081-a97d-2fd80dab9174,ad478070-3c46-4a28-8fa3-0e50d87a1f79,2022-01-01 01:01:48,10395,NaN,0,NaN,NaN,NaN,0.000000
4,cc7b62bf-0a75-4580-bd9d-99281ff7f1ed,8ce6d296-edf3-4c1e-8d2c-a571fb7f9d93,a8382e93-02f6-4c54-ae97-1752b37dc9e2,2022-01-01 01:19:20,603,NaN,0,NaN,NaN,NaN,0.068966


In [50]:
#Buyer fraud rate
merged_sorted['buyer_fraud_rate'] = (
    merged_sorted.groupby('buyer_id')['is_fraud']
    .expanding().mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

In [51]:
merged_sorted['buyer_fraud_rate'] = merged_sorted['buyer_fraud_rate'].fillna(0)

In [52]:
merged_sorted.head(5)

,id,merchant_id,buyer_id,transaction_timestamp,payment_amount,chargeback_timestamp,is_fraud,country_x,country_y,category,merchant_fraud_rate,buyer_fraud_rate
0,0ef41176-3ec5-4ab6-9c37-bb793ec75e54,977ce629-3cbe-43ec-878f-cf1c2aebe85d,e5737c15-ffcd-4111-a0b0-d6746abf7ab7,2022-01-01 00:22:09,3469,2022-01-07 00:30:10.855945,1,NaN,NaN,NaN,0.059829,0.000000
1,596a94fa-7398-4b30-a0d8-cc518a8485c0,951bce36-de7c-4081-a97d-2fd80dab9174,ea546275-cf52-4ea7-80c9-f243bfeea139,2022-01-01 00:32:11,321,NaN,0,NaN,NaN,NaN,0.111111,0.000000
2,1227adb3-b15b-46f8-9d15-71d0071019b8,af9ccc42-0a10-4acb-9a78-2ca0fe889cd9,b3ed51d9-8dbd-4e46-bfd3-b31217e8a033,2022-01-01 00:55:13,622,2022-01-13 11:51:50.293075,1,NaN,NaN,NaN,0.031250,0.000000
3,1d6c373e-167a-48bd-a748-a48ca76a5afe,951bce36-de7c-4081-a97d-2fd80dab9174,ad478070-3c46-4a28-8fa3-0e50d87a1f79,2022-01-01 01:01:48,10395,NaN,0,NaN,NaN,NaN,0.000000,0.000000
4,cc7b62bf-0a75-4580-bd9d-99281ff7f1ed,8ce6d296-edf3-4c1e-8d2c-a571fb7f9d93,a8382e93-02f6-4c54-ae97-1752b37dc9e2,2022-01-01 01:19:20,603,NaN,0,NaN,NaN,NaN,0.068966,0.333333


In [53]:
#Printing the results
sum_merchant = merged_sorted['merchant_fraud_rate'].sum()
sum_buyer = merged_sorted['buyer_fraud_rate'].sum()
total = sum_merchant + sum_buyer

In [54]:
print(f"sum of average merchant fraud rates: {sum_merchant}")
print(f" sum of average buyer fraud rates:{sum_buyer}")
print(f"Total: {total}")


sum of average merchant fraud rates: 1154.0661044293806
 sum of average buyer fraud rates:1161.0062506937506
Total: 2315.0723551231313
